In [38]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [39]:
corpus = [
    "the sun rises in the east",
    "the moon shines at night",
    "the stars glow in the sky",
    "the sky looks beautiful today",
    "the sun shines brightly"
]

In [40]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

total_words = len(tokenizer.word_index) + 1

In [41]:
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

In [42]:
max_len = max(len(x) for x in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [43]:
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.query = tf.keras.layers.Dense(embed_dim)
        self.key = tf.keras.layers.Dense(embed_dim)
        self.value = tf.keras.layers.Dense(embed_dim)
    def call(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = tf.matmul(Q, K, transpose_b=True)
        weights = tf.nn.softmax(scores)

        return tf.matmul(weights, V)

In [44]:
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.attention = SelfAttention(embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(embed_dim)
        ])
    def call(self, x):
        attn_output = self.attention(x)
        x = x + attn_output
        ffn_output = self.ffn(x)
        return x + ffn_output

In [45]:
embed_dim = 32
num_heads = 2
ff_dim = 64

inputs = tf.keras.layers.Input(shape=(max_len-1,))
x = tf.keras.layers.Embedding(total_words, embed_dim)(inputs)
x = EncoderBlock(embed_dim, num_heads, ff_dim)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)

outputs = tf.keras.layers.Dense(total_words, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

TypeError: EncoderBlock.__init__() takes 2 positional arguments but 4 were given

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.fit(X, y,epochs=200)

In [ ]:
test = ["machine learning is "]
test_seq = tokenizer.texts_to_sequences(test)
test_pad = pad_sequences(test_seq, maxlen=max_len, padding="post")
prediction = model.predict(test_pad)

print(prediction)